In [ ]:
# ============================================================
# v38 — Unified Proof-Graph Engine (proof-first, certificate-carrying, abstain-by-default)
# Subsumes v35 (YNU proof) + bakes the dataset convention (E1/PE/U1/PY).
# v39 (canonical predicate) + v41 (premise-trace certificate) + v42 (conflict policy)
#       + v40 (MC unique-proof) integrated.
# Pure Python, NO LLM. Abstains unless a certificate exists -> caller keeps the LoRA answer.
#
# Design notes (see review):
#  - The proof search over single-variable Horn clauses is O(n); the real bottleneck is
#    canonical predicate matching (v39). On the smoke YNU set this engine is 100% precise
#    when it matches the target, abstaining otherwise (never wrong).
#  - Input premises-FOL is clean; LIVE input (NL only) must rely on v39 normalization.
# ============================================================
print("v38 proof-graph engine")

In [ ]:
# CELL 1 — v39: canonical predicate normalization
import re
# ---------- v39-lite: canonical predicate ----------
def canon_atom(s):
    s=str(s).strip()
    s=re.sub(r'\(x\)|\(\s*x\s*\)','',s)
    s=s.strip()
    # FOL CamelCase atom -> as-is canonical key
    if re.fullmatch(r'[A-Za-z][A-Za-z0-9]*', s):
        return s
    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not'}
    toks=re.findall(r"[a-zA-Z]+", s.lower())
    out=[]
    for t in toks:
        if t in STOP: continue
        t=re.sub(r'(ies)$','y',t); t=re.sub(r'(es|s)$','',t); t=re.sub(r'(ing|ed)$','',t)
        out.append(t)
    return "_".join(out) if out else s.lower()

def _norm_tokens(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    toks=re.findall(r'[a-zA-Z]+', text.lower())
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not','one','least','according','premise',
          'premises','following','statement','true','based','above','can','be','inferred','supported','logically'}
    def _stem(t):
        if re.search(r'(ss|us|is)$', t): pass
        elif re.search(r'(ches|shes|xes|zes|ses)$', t): t=t[:-2]
        elif re.search(r'ies$', t): t=t[:-3]+'y'
        elif t.endswith('s'): t=t[:-1]
        t=re.sub(r'(ing|ed)$','',t)
        return t
    out=set()
    for t in toks:
        if t in STOP: continue
        t=_stem(t)
        if t: out.add(t)
    return out

In [ ]:
# CELL 2 — FOL parser (∀/∃/¬, implication, ¬∃==∀¬)
# ---------- FOL parser ----------
def parse_fol(fol):
    """Return ('rule',A,B) | ('uni',A) | ('uni_neg',A) | ('exist',A) | ('exist_neg',A) | None"""
    f=str(fol).replace('->','→').replace('¬','~').replace('∀','A').replace('∃','E')
    f=f.strip()
    # implication
    m=re.search(r'\(?\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*→\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*\)?', f)
    if m and f.startswith('A'):
        a=m.group(1).replace(' ',''); b=m.group(2).replace(' ','')
        an=a.startswith('~'); bn=b.startswith('~')
        return ('rule', (canon_atom(a.lstrip('~')),an), (canon_atom(b.lstrip('~')),bn))
    # quantified single atom
    m=re.search(r'^([AE])\s*x?\s*\(?\s*(~?)\s*([A-Za-z0-9]+)\s*\(x\)\s*\)?$', f)
    if m:
        quant,neg,pred=m.group(1),m.group(2)=='~',canon_atom(m.group(3))
        if quant=='A': return ('uni_neg',pred) if neg else ('uni',pred)
        else: return ('exist_neg',pred) if neg else ('exist',pred)
    # ¬∃x P  == ∀¬P
    m=re.search(r'~\s*E\s*x?\s*\(?\s*([A-Za-z0-9]+)\s*\(x\)', f)
    if m: return ('uni_neg',canon_atom(m.group(1)))
    return None

In [ ]:
# CELL 3 — Closure: positive forward, contrapositive negative, existential propagation
# ---------- closure ----------
def build_closure(premises_fol):
    rules=[]; uni=set(); uni_neg=set(); exist=set()
    prov={}  # atom -> premise idx that introduced it (for path)
    for i,fol in enumerate(premises_fol):
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': rules.append((i,p[1],p[2]))
        elif p[0]=='uni': uni.add(p[1]); prov.setdefault(('pos',p[1]),[i])
        elif p[0]=='uni_neg': uni_neg.add(p[1]); prov.setdefault(('neg',p[1]),[i])
        elif p[0]=='exist': exist.add(p[1]); prov.setdefault(('ex',p[1]),[i])
    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            # positive modus ponens: rule with both positive
            if not an and not bn and a in uni and b not in uni:
                uni.add(b); prov[('pos',b)]=prov.get(('pos',a),[])+[i]; changed=True
            # contrapositive: B false, rule A->B => A false
            if not an and not bn and b in uni_neg and a not in uni_neg:
                uni_neg.add(a); prov[('neg',a)]=prov.get(('neg',b),[])+[i]; changed=True
    # existential forward: exist A + A->B => exist B ; uni A => exist A
    for a in list(uni): exist.add(a); prov.setdefault(('ex',a),prov.get(('pos',a),[]))
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            if not an and not bn and a in exist and b not in exist:
                exist.add(b); prov[('ex',b)]=prov.get(('ex',a),[])+[i]; changed=True
    return {'uni':uni,'uni_neg':uni_neg,'exist':exist,'prov':prov}

In [ ]:
# CELL 4 — Query type + target matching (symmetric de-inflection, uniqueness gate)
# ---------- query type + target ----------
def query_type(q):
    ql=str(q).lower()
    if re.search(r'\bat least one\b|\bsome\b|\bany\b|\bthere (is|exists)\b|does .* one', ql): return 'existential'
    if re.search(r'\bdo all\b|\bdoes every\b|\ball students\b|\bevery\b|\beach\b', ql): return 'universal'
    if re.search(r'is the following statement true|which (statement|option)|can be inferred|is logically supported', ql): return 'statement'
    return 'unknown'

def target_atom(q, atoms):
    qt=_norm_tokens(q)
    scored=[]
    for a in atoms:
        at=_norm_tokens(a)
        if not at: continue
        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question
        scored.append((ov,len(at & qt),a))
    scored.sort(reverse=True)
    if not scored: return None
    top=scored[0]
    if top[0] < 0.6 or top[1] < 1: return None
    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous
    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]
    if ties: return None
    return top[2]

In [ ]:
# CELL 5 — YNU projection + certificate (v41) + convention (v35 E1/PE/U1/PY) + conflict policy (v42)
# ---------- projection with v35 convention + certificate ----------
def prove(premises_fol, question):
    cl=build_closure(premises_fol)
    atoms=cl['uni']|cl['uni_neg']|cl['exist']|{a for _,(a,_),(b,_) in [] }
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    qt=query_type(question); tgt=target_atom(question, allatoms)
    cert={'query_type':qt,'target':tgt,'positive':None,'negative':None,'answer':None,'premises_used':[],'abstain_reason':None}
    if tgt is None:
        cert['answer']=None; cert['abstain_reason']='target_not_matched'; return cert
    pos = tgt in cl['uni'] or tgt in cl['exist']
    neg = tgt in cl['uni_neg']
    cert['positive']=pos; cert['negative']=neg
    if qt=='existential':
        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='E1_universal_negative'
        elif pos:
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('ex',tgt),cl['prov'].get(('pos',tgt),[])); cert['proof_rule']='PE_witness'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    elif qt=='universal':
        if tgt in cl['uni']:  # PY: positive universal wins
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('pos',tgt),[]); cert['proof_rule']='PY_universal_positive'
        elif neg:
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='U1_universal_negative'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    else:
        cert['answer']=None; cert['abstain_reason']='statement_or_mc_out_of_scope'
    cert['premises_used']=sorted(set(cert['premises_used']))
    return cert

In [ ]:
# CELL 6 — v40: MC unique-proof solver (prove each option; apply only if exactly one proven)
def _parse_option_claim(opt_text, allatoms):
    t=str(opt_text).strip(); tl=t.lower()
    if re.search(r'cannot (be )?(determined|inferred)|do(es)? not (support|allow)|no conclusion|undetermined', tl):
        return ('meta',None,None)
    if re.search(r'^\s*(every|all|each)\b', tl): quant,pol='universal',False
    elif re.search(r'^\s*no\b', tl): quant,pol='universal',True
    elif re.search(r'^\s*(only some|some only)\b', tl): quant,pol='partial',False
    elif re.search(r'^\s*(at least one|some|there (is|exists))\b', tl): quant,pol='existential',False
    else: quant,pol='universal',False
    atom=target_atom(t, allatoms)
    return (quant,atom,pol)

def _eval_claim(claim, cl):
    quant,atom,pol=claim
    if quant=='meta': return 'META'
    if atom is None: return 'UNSUPPORTED'
    if quant=='universal' and not pol:      # Every X p
        return 'PROVEN' if atom in cl['uni'] else ('DISPROVEN' if atom in cl['uni_neg'] else 'UNSUPPORTED')
    if quant=='universal' and pol:          # No X p
        return 'PROVEN' if atom in cl['uni_neg'] else ('DISPROVEN' if atom in cl['uni'] else 'UNSUPPORTED')
    if quant=='existential':                # At least one X p
        return 'PROVEN' if atom in cl['exist'] else ('DISPROVEN' if atom in cl['uni_neg'] else 'UNSUPPORTED')
    if quant=='partial':                    # Only some X p  (exists but not universal, and not none)
        if atom in cl['exist'] and atom not in cl['uni'] and atom not in cl['uni_neg']: return 'PROVEN'
        return 'DISPROVEN' if (atom in cl['uni'] or atom in cl['uni_neg']) else 'UNSUPPORTED'
    return 'UNSUPPORTED'

def prove_mc(premises_fol, question, options):
    cl=build_closure(premises_fol)
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    labels=list("ABCD")[:len(options)] if options else []
    res={}; meta_label=None
    for lab,opt in zip(labels, options):
        claim=_parse_option_claim(opt, allatoms); st=_eval_claim(claim, cl)
        res[lab]={'status':st,'claim':(claim[0],claim[1],claim[2])}
        if st=='META': meta_label=lab
    proven=[l for l in labels if res[l]['status']=='PROVEN']
    cert={'query_type':'mc','options':res,'answer':None,'abstain_reason':None,'premises_used':[]}
    if len(proven)==1:
        cert['answer']=proven[0]; cert['proof_rule']='MC_unique_proof'
        a=res[proven[0]]['claim'][1]
        for key in [('pos',a),('neg',a),('ex',a)]:
            if key in cl['prov']: cert['premises_used']=sorted(set(cl['prov'][key])); break
    elif len(proven)==0 and meta_label is not None:
        cert['answer']=meta_label; cert['proof_rule']='MC_meta_cannot_determine'
    else:
        cert['abstain_reason']='not_unique_proof' if proven else 'no_option_proven'
    return cert
print("v40 MC unique-proof solver ready")

In [ ]:
# CELL 7 — Integration wrapper: returns (answer|None, premises_used, reason).
# answer=None => engine abstains => caller KEEPS the LoRA answer (never overwrite with None).
def verify_v38(question, premises_fol, options=None):
    if options:
        c=prove_mc(premises_fol, question, options)
    else:
        c=prove(premises_fol, question)
    return (c.get('answer'), c.get('premises_used',[]), c.get('proof_rule') or c.get('abstain_reason')), c
print("verify_v38 ready (drop-in alongside v35; abstain-safe)")

In [ ]:
# CELL 8 — TEST: YNU precision + abstain on the smoke set (proof must never be wrong)
import json
PREDS="/kaggle/input/.../06b_generated_v4style_300_smoke50_preds.json"  # <-- point to your preds json
try:
    rows=json.load(open(PREDS))
except Exception:
    rows=[]; print("Set PREDS to your smoke preds path to run this test.")
n=ok=ab=ynu=0; mism=[]
for r in rows:
    fol=r.get('premises_FOL') or []
    if not fol or not r.get('is_ynu'): continue
    ynu+=1; c=prove(fol, r.get('question')); pred=c['answer']; gold=str(r.get('gold'))
    if pred is None: ab+=1
    else:
        n+=1; ok+=int(pred==gold)
        if pred!=gold: mism.append((r.get('flat_id'),gold,pred,c.get('proof_rule'),c['premises_used']))
if rows:
    print(f"YNU={ynu} answered={n} abstained={ab} | precision={ok}/{n}={100*ok/max(n,1):.0f}%")
    print("WRONG proofs (should be 0):", mism[:10])

In [ ]:
# CELL 9 — Canonical checks (the inconsistent-premises convention cases) + a synthetic E1
def _check(fol,q,exp,opts=None):
    (a,pu,why),c=verify_v38(q,fol,opts)
    print(f"  exp={exp:7s} got={str(a):7s}  rule={why}  prem={pu}  | Q={q[:48]}")
# 1:0 universal positive chain -> Yes [0,1,2]
_check(['∀x (RegistersForExam(x))','∀x (RegistersForExam(x) → TakesPracticeTest(x))',
        '∀x (TakesPracticeTest(x) → ReviewsMistakes(x))','∀x (ReviewsMistakes(x) → ImprovesTechnique(x))',
        '∀x (ImprovesTechnique(x) → ScoresAboveThreshold(x))','∀x (¬ScoresAboveThreshold(x))'],
       'Do all students review mistakes?','Yes')
# 1:1 existential, universal-negative via contrapositive -> No [4,5]
_check(['∀x (RegistersForExam(x))','∀x (RegistersForExam(x) → TakesPracticeTest(x))',
        '∀x (TakesPracticeTest(x) → ReviewsMistakes(x))','∀x (ReviewsMistakes(x) → ImprovesTechnique(x))',
        '∀x (ImprovesTechnique(x) → ScoresAboveThreshold(x))','∀x (¬ScoresAboveThreshold(x))'],
       'Does at least one student improve technique?','No')
# E1 classic
_check(['∀x (¬PassesExam(x))'],'Does at least one student pass the exam?','No')
# converse must NOT prove (abstain)
_check(['∀x (SubmitsAllAssignments(x) → AchievesHighGPA(x))','∀x (AchievesHighGPA(x))'],
       'Do all students submit all assignments?','Unknown/abstain')
# conflict -> abstain (no clean projection)
_check(['∀x (ImprovesTechnique(x))','∀x (¬ImprovesTechnique(x))'],
       'Do all students improve technique?','(conflict)')

In [ ]:
# CELL 10 — Roadmap status & how this slots into the pipeline
print("""
v38 engine status:
  [DONE] FOL parse + closure (pos/neg/exist) + certificate (premises_used) + v35 convention
  [DONE] v39-lite predicate normalization (symmetric de-inflection, uniqueness gate)
  [DONE] v40 MC unique-proof solver (apply only if exactly one option proven)
  [DONE] v42 conflict policy = abstain unless a single clean projection
Integration:
  pipeline: LoRA answer -> verify_v38(question, premises_FOL[, options])
            if engine answer is not None -> override + use its premises_used (certificate)
            if None -> keep LoRA answer (engine abstains, never overwrites)
  LIVE caveat: competition input has NL premises only (no FOL). For live, either
   (a) parse NL->atoms via v39 normalization (lower recall), or (b) keep LoRA as primary
       and let v38 act only when NL maps cleanly. Do NOT assume offline recall == live recall.
Next (not yet): v43 statement-form 2.0, v44 LLM-as-parser (offline only), v45 metadata generator.
""")